In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
import copy
import seaborn as sns
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset, ConcatDataset

#import umap
#import hdbscan
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split, KFold
from sklearn.metrics import adjusted_rand_score
from tqdm import tqdm
from sklearn.manifold import TSNE

ModuleNotFoundError: No module named 'torch'

In [ ]:
def read_unequal_table(txt_file, num_columns=383, filler=''):
    """Read table from file."""
    table = []
    with open(txt_file) as file:
        for row in file:
            values = row.split()
            if len(values) > num_columns:
                raise ValueError('Number of values more than num_columns')
            elif len(values) < num_columns:
                diff = num_columns - len(values)
                padding = [filler]*diff
                values = values + padding
            else:
                pass
            table.append(values)
    return np.array(table)

In [ ]:
def read_norm_spectra(txt_file):
    """Read the saved set of normalized spectra"""
    alldata = read_unequal_table(txt_file)
    srcids = alldata[:, 0].astype(str)
    normed_spectra = alldata[:, 1:-2].astype(float)
    netcounts = alldata[:, -2].astype(float)
    src_class = alldata[:, -1].astype(str)
    return srcids, normed_spectra, netcounts, src_class

In [4]:
def refine_labels(label_array):
    """Refine labels to 4 classes."""
    refined_labels = np.full(len(label_array), '', dtype='<U16')
    for i, label in enumerate(label_array):
        if label == '':
            continue
        if label == 'AGN' or label == 'CV':
            refined_labels[i] = label
        elif label == 'LM-STAR' or label == 'HM-STAR' or label == 'YSO':
            refined_labels[i] = 'STAR'
        else:
            refined_labels[i] = 'NS/BH'
    return refined_labels

In [6]:
pn_srcids, pn_normspec, pn_counts, pn_class = read_norm_spectra('../Data/Brightpn_id_normspec_counts_label.txt')

In [7]:
mos_srcids, mos_normspec, mos_counts, mos_class = read_norm_spectra('../Data/Brightmos_id_normspec_counts_label.txt')

In [8]:
def check_spectra(normspec, refinedclass):
    """Check the normalized spectra of different classes."""
    en_bins = np.linspace(0.5, 10, len(normspec[0]))
    classes_unique = np.unique(refinedclass)
    #plt.figure(figsize=(20, 12))
    plt.xlabel('Energy [keV]')
    plt.ylabel('Normalized counts [/bin]')
    plt.xscale('log')
    plt.xlim(0.5, 10.0)
    for classes in classes_unique:
        if classes != '':
            plt.plot(en_bins,
                     np.mean(normspec[refinedclass == classes], axis=0), label=classes)
    plt.legend()
    plt.show

In [11]:
pn_class_refined = refine_labels(pn_class)
mos_class_refined = refine_labels(mos_class)

In [10]:
print(f"Number of AGN: {len(np.where(pn_class_refined == 'AGN')[0])}")
print(f"Number of STAR: {len(np.where(pn_class_refined == 'STAR')[0])}")
print(f"Number of NS: {len(np.where(pn_class_refined == 'NS/BH')[0])}")
print(f"Number of CVs: {len(np.where(pn_class_refined == 'CV')[0])}")

Number of AGN: 1339
Number of STAR: 1193
Number of NS: 99
Number of CVs: 99


In [12]:
print(f"Number of AGN: {len(np.where(mos_class_refined == 'AGN')[0])}")
print(f"Number of STAR: {len(np.where(mos_class_refined == 'STAR')[0])}")
print(f"Number of NS: {len(np.where(mos_class_refined == 'NS/BH')[0])}")
print(f"Number of CVs: {len(np.where(mos_class_refined == 'CV')[0])}")

Number of AGN: 720
Number of STAR: 621
Number of NS: 81
Number of CVs: 81


In [135]:
pn_labelled_srcids = pn_srcids[pn_class_refined != '']
mos_labelled_srcids = mos_srcids[mos_class_refined != '']

In [66]:
pn_agns_star_ids = pn_srcids[np.logical_or(
    pn_class_refined == 'AGN', pn_class_refined == 'STAR')]
mos_agns_star_ids = mos_srcids[np.logical_or(
    mos_class_refined == 'AGN', mos_class_refined == 'STAR')]

In [136]:
pn_mos_labelled_ids = np.array(list(set(pn_labelled_srcids) |
                                    set(mos_labelled_srcids)))

In [137]:
len(pn_mos_labelled_ids)

2975

In [138]:
np.random.shuffle(pn_mos_labelled_ids)
pn_mos_labelled_ids

array(['208650507010002', '206050006010005', '207207003010038', ...,
       '208022008010003', '202033606010038', '201096608010017'],
      dtype='<U15')

In [139]:
random_nums = np.arange(len(pn_mos_labelled_ids))
np.random.shuffle(random_nums)
random_nums

array([1118, 1809, 2664, ..., 2399, 1326, 2681])

In [140]:
test_labelled_ids = pn_mos_labelled_ids[random_nums[:int(0.2*len(random_nums))]]
train_labelled_ids= pn_mos_labelled_ids[random_nums[int(0.2*len(random_nums)):]]

In [141]:
len(train_labelled_ids)

2380

In [142]:
def get_pn_mos_indices(src_id_list, pn_ids, mos_ids):
    pn_indices = []
    mos_indices = []
    for src_id in src_id_list:
        mask_pn = (pn_ids == src_id)
        mask_mos = (mos_ids == src_id)
        if mask_pn.sum() > 0:
            pn_indices.append(np.where(mask_pn)[0][0])
        if mask_mos.sum() > 0:
            mos_indices.append(np.where(mask_mos)[0][0])
        if mask_pn.sum() == 0 and mask_mos.sum() == 0:
            print(f"Srcid {src_id} not found in both PN and MOS")
            break

    return pn_indices, mos_indices

In [102]:
np.savetxt('PN_MOS_test_ids.txt', test_labelled_ids, fmt='%s')

In [103]:
np.savetxt('PN_MOS_train_ids.txt', train_labelled_ids, fmt='%s')

In [151]:
pn_test_indices_all, mos_test_indices_all = get_pn_mos_indices(
    test_labelled_ids, pn_labelled_srcids, mos_labelled_srcids)

In [152]:
pn_train_indices_all, mos_train_indices_all = get_pn_mos_indices(
    train_labelled_ids, pn_labelled_srcids, mos_labelled_srcids)

In [159]:
np.max(pn_train_indices_all)

2728

In [146]:
len(mos_train_indices_all)

1203

In [108]:
len(test_labelled_ids)

550

In [154]:
np.savetxt('Test_PN_indices_forall_labelled.txt', pn_test_indices_all)

In [155]:
np.savetxt('Test_MOS_indices_forall_labelled.txt', mos_test_indices_all)

In [156]:
np.savetxt('Train_PN_indices_forall_labelled.txt', pn_train_indices_all)

In [157]:
np.savetxt('Train_MOS_indices_forall_labelled.txt', mos_train_indices_all)

In [163]:
(pn_class_refined[pn_class_refined != ''][pn_train_indices_all] == 'CV').sum()

82

In [114]:
pn_train_indices[1]

85

In [120]:
pn_normspec[np.logical_or(pn_class_refined=='AGN', pn_class_refined=='STAR')][2220]

array([ 1.41836689e-02,  1.28571938e-02,  2.56366279e-02,  1.10892186e-02,
        2.71634258e-02,  3.13431738e-02,  4.28768544e-02,  2.47127732e-02,
        2.55160392e-02,  4.91056220e-02,  3.54023332e-02,  4.95072551e-02,
        5.04719642e-02,  3.85765177e-02,  4.53683614e-02,  5.75049852e-02,
        3.54422003e-02,  4.02647587e-02,  3.41964468e-02,  1.80813852e-02,
        3.61258651e-02,  2.92134327e-02,  8.83888882e-03,  2.29438107e-02,
        1.56705997e-02,  1.86843284e-02,  1.18934720e-02,  1.02450981e-02,
        6.34738177e-03,  2.85129846e-03,  5.82516008e-03,  1.20937948e-02,
        8.43725578e-03,  8.07548986e-03,  1.22951050e-02, -3.63740432e-04,
        7.35195802e-03,  1.67957646e-02,  2.73070982e-03,  1.72514632e-03,
        7.99476834e-03, -2.93596900e-03,  8.42145968e-04,  1.76600070e-03,
        8.43725578e-03,  4.94117247e-03,  2.81143134e-03, -2.73465885e-03,
        4.17777350e-03,  2.93201998e-03,  2.97188710e-03,  3.93659622e-03,
        3.85587471e-03,  

In [119]:
np.logical_or(pn_class_refined=='AGN', pn_class_refined=='STARS').sum()

1339

In [127]:
len(np.where(pn_class_refined[pn_test_indices] == 'AGN')[0])

285

In [130]:
pn_srcids[pn_train_indices[50]]

'208039507010004'

In [131]:
mos_srcids[mos_train_indices[50]]

'201532201010002'

In [132]:
len(mos_train_indices)

1071

In [134]:
len(np.loadtxt('Train_MOS_indices_foragnstars.txt'))

1071

# Analyzing combined PN MOS data

In [164]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4, dropout_rate=0.2):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=8, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(8)
        self.pool1 = nn.MaxPool1d(2)  # 380 → 190
        self.drop1 = nn.Dropout(dropout_rate)  # <-- dropout after first conv block

        self.conv2 = nn.Conv1d(8, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(16)
        self.pool2 = nn.MaxPool1d(2)  # 190 → 95
        self.drop2 = nn.Dropout(dropout_rate)  # <-- dropout after second conv block

        # final feature size = 16 * 95 = 1520
        self.fc1 = nn.Linear(16 * 95, 64)
        self.drop3 = nn.Dropout(dropout_rate)  # <-- dropout before final layer
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.drop1(x)
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.drop2(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.drop3(x)
        return self.fc2(x)

In [173]:
model_pl_pn = torch.load('SimpleCNN_reg_leePsuedolabel2.pt')
model_pl_mos = torch.load('SimpleCNN_reg_leePsuedolabel_MOS2.pt')
model_pl_pn.eval()
model_pl_pn.eval()

SimpleCNN(
  (conv1): Conv1d(1, 8, kernel_size=(7,), stride=(1,), padding=(3,))
  (bn1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout(p=0.0, inplace=False)
  (conv2): Conv1d(8, 16, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout(p=0.0, inplace=False)
  (fc1): Linear(in_features=1520, out_features=64, bias=True)
  (drop3): Dropout(p=0.0, inplace=False)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)

In [166]:
test_pn_indices =np.loadtxt('Test_PN_indices.txt')
test_mos_indices = np.loadtxt('Test_MOS_indices.txt')
print(len(test_pn_indices), len(test_mos_indices))
print(np.max(test_pn_indices), np.max(test_mos_indices))

509 270
60282.0 27900.0


In [169]:
test_pn_data = pn_normspec[test_pn_indices.astype(int)]
test_mos_data = mos_normspec[test_mos_indices.astype(int)]
test_pn_labels = pn_class_refined[test_pn_indices.astype(int)]
test_mos_labels = mos_class_refined[test_mos_indices.astype(int)]
test_pn_srcids = pn_srcids[test_pn_indices.astype(int)]
test_mos_srcids = mos_srcids[test_mos_indices.astype(int)]

In [172]:
test_pn_labels[test_pn_labels =='AGN'] = 0
test_pn_labels[test_pn_labels == 'STAR'] = 1
test_mos_labels[test_mos_labels =='AGN'] = 0
test_mos_labels[test_mos_labels == 'STAR'] = 1

In [178]:
test_mos_labels

array(['0', '1', '1', '0', '0', '0', '1', '0', '0', '1', '1', '1', '1',
       '1', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0',
       '0', '1', '1', '0', '1', '0', '1', '0', '0', '1', '0', '0', '1',
       '1', '0', '0', '0', '0', '1', '1', '0', '0', '1', '0', '0', '0',
       '1', '1', '0', '0', '1', '1', '1', '0', '1', '0', '1', '0', '1',
       '0', '0', '1', '0', '1', '1', '1', '0', '0', '0', '0', '1', '1',
       '0', '1', '1', '1', '0', '0', '1', '1', '0', '0', '0', '1', '0',
       '0', '0', '1', '1', '1', '0', '0', '0', '1', '0', '0', '1', '0',
       '0', '0', '1', '1', '0', '1', '0', '0', '0', '1', '0', '0', '1',
       '0', '1', '0', '1', '1', '0', '0', '0', '1', '0', '0', '0', '1',
       '1', '1', '0', '0', '0', '0', '1', '1', '1', '0', '1', '1', '0',
       '1', '0', '0', '0', '1', '0', '1', '1', '0', '0', '1', '0', '0',
       '0', '0', '0', '1', '1', '0', '1', '0', '1', '0', '0', '1', '0',
       '1', '0', '0', '1', '0', '0', '0', '0', '0', '1', '1', '1

In [186]:
def evaluate_conf(model, loader, device):
    model.eval()
    all_preds = np.array([])
    all_probs_agn = np.array([])
    all_probs_stars = np.array([])
    with torch.no_grad():
        for xb in loader:
            xb = xb[0].to(device)
            probs = torch.softmax(model(xb), dim=1)
            conf, preds = torch.max(probs, dim=1)
            all_preds = np.append(all_preds.copy(), np.array(preds))
            # all_confs = np.append(all_confs.copy(), np.array(conf))
            all_probs_agn = np.append(all_probs_agn.copy(),
                                      np.array(probs[:, 0]))
            all_probs_stars = np.append(all_probs_stars.copy(),
                                        np.array(probs[:, 1]))
            
    return all_preds, all_probs_agn, all_probs_stars

In [183]:
test_pn_ds = TensorDataset(
    torch.tensor(test_pn_data, dtype=torch.float32).unsqueeze(1),
    torch.tensor(test_pn_labels.astype(int), dtype=torch.long))
test_mos_ds = TensorDataset(
    torch.tensor(test_mos_data, dtype=torch.float32).unsqueeze(1),
    torch.tensor(test_mos_labels.astype(int), dtype=torch.long))
test_pn_loader = DataLoader(test_pn_ds, batch_size=64, shuffle=False)
test_mos_loader = DataLoader(test_mos_ds, batch_size=64, shuffle=False)

In [180]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [187]:
test_pn_preds, test_pn_agnprob, test_pn_starprob = evaluate_conf(
    model_pl_pn, test_pn_loader, device)
test_mos_preds, test_mos_agnprob, test_mos_starprob = evaluate_conf(
    model_pl_mos, test_mos_loader, device)

In [188]:
test_pn_srcid_conf = np.column_stack([test_pn_srcids, test_pn_preds,
                                      test_pn_agnprob, test_pn_starprob])
test_mos_srcid_conf = np.column_stack([test_mos_srcids, test_mos_preds,
                                       test_mos_agnprob, test_mos_starprob])

In [189]:
test_srcids_all = np.array(list(
    set(test_pn_srcids) | set(test_mos_srcids)))

In [244]:
test_pnmos_preds = np.zeros(len(test_srcids_all), dtype=int)
test_pnmos_confs = np.zeros(len(test_srcids_all), dtype=float)
test_pnmos_labels = np.zeros(len(test_srcids_all), dtype=int)

In [245]:
pn_pred_swithces = []
mos_pred_switches = []

In [246]:
for i, srcid in enumerate(test_srcids_all):
    pn_index = np.where(test_pn_srcids == srcid)[0]
    mos_index = np.where(test_mos_srcids == srcid)[0]
    if len(pn_index) == 1 and len(mos_index) == 1:
        #Sanity check
        if test_pn_labels[pn_index] != test_mos_labels[mos_index]:
            print('Something srsly wrong')
        else:
            test_pnmos_labels[i] = int(test_pn_labels[pn_index])
        prob_ratio = ((test_pn_starprob[pn_index[0]] *
                       test_mos_starprob[mos_index[0]]) /
                      (test_pn_agnprob[pn_index[0]] *
                       test_mos_agnprob[mos_index[0]]))
        if prob_ratio >= 1.0:
            test_pnmos_preds[i] = 1
            test_pnmos_confs[i] = prob_ratio/(1.0 + prob_ratio)
        else:
            test_pnmos_preds[i] = 0
            test_pnmos_confs[i] = 1/(1.0 + prob_ratio)
        if test_pnmos_preds[i] != test_pn_preds[pn_index[0]]:
            pn_pred_swithces.append(i)
        if test_pnmos_preds[i] != test_mos_preds[mos_index[0]]:
            mos_pred_switches.append(i)
    elif len(pn_index) == 1:
        test_pnmos_labels[i] = int(test_pn_labels[pn_index])
        if test_pn_agnprob[pn_index] >= test_pn_starprob[pn_index[0]]:
            test_pnmos_preds[i] = 0
            test_pnmos_confs[i] = test_pn_agnprob[pn_index[0]]
        else:
            test_pnmos_preds[i] = 1
            test_pnmos_confs[i] = test_pn_starprob[pn_index[0]]
    elif len(mos_index) == 1:
        test_pnmos_labels[i] = int(test_mos_labels[mos_index[0]])
        if test_mos_agnprob[mos_index] >= test_mos_starprob[mos_index[0]]:
            test_pnmos_preds[i] = 0
            test_pnmos_confs[i] = test_mos_agnprob[mos_index[0]]
        else:
            test_pnmos_preds[i] = 1
            test_pnmos_confs[i] = test_mos_starprob[mos_index[0]]
    else:
        print('Src id not found')

In [247]:
def get_stats(preds, labels, conf=None, threshold=0.9):
    if conf is None:
        conf = np.ones(len(preds), dtype=bool)
    preds0 = np.logical_and(preds == 0, conf >= threshold).sum()
    preds1 = np.logical_and(preds == 1, conf >= threshold).sum()
    labels0 = (labels == 0).sum()
    labels1 = (labels == 1).sum()
    corr0 = np.logical_and(np.logical_and(preds==0, labels==0),
                           conf >= threshold).sum()
    corr1 = np.logical_and(np.logical_and(preds==1, labels==1),
                           conf >= threshold).sum()
    return ((corr0 + corr1)/len(preds[conf>=threshold]),
            corr0/labels0, corr1/labels1,
            corr0/preds0, corr1/preds1)

In [257]:
get_stats(test_pn_preds, test_pn_labels.astype(int))

(0.931237721021611,
 0.9508771929824561,
 0.90625,
 0.928082191780822,
 0.9354838709677419)

In [258]:
get_stats(test_mos_preds, test_mos_labels.astype(int))

(0.937037037037037,
 0.9551282051282052,
 0.9122807017543859,
 0.9371069182389937,
 0.9369369369369369)

In [259]:
get_stats(test_pnmos_preds, test_pnmos_labels)

(0.9309090909090909,
 0.9451612903225807,
 0.9125,
 0.9331210191082803,
 0.9279661016949152)

In [260]:
get_stats(test_pnmos_preds, test_pnmos_labels)

(0.9309090909090909,
 0.9451612903225807,
 0.9125,
 0.9331210191082803,
 0.9279661016949152)

In [226]:
len(mos_pred_switches)

9

In [227]:
np.where(test_pnmos_preds != test_pnmos_labels)

(array([  4,   7,   9,  13,  17,  19,  21,  26,  28,  29,  31,  35,  37,
         40,  49,  55,  57,  59,  60,  61,  63,  65,  71,  76,  78,  84,
         92,  93,  96,  97,  98, 100, 104, 106, 107, 108, 109, 110, 116,
        117, 120, 123, 127, 136, 137, 139, 140, 142, 144, 145, 147, 150,
        152, 154, 156, 158, 160, 169, 176, 178, 184, 186, 187, 191, 194,
        198, 212, 222, 223, 233, 239, 245, 252, 253, 257, 258, 259, 264,
        267, 279, 287, 295, 299, 301, 304, 306, 308, 316, 323, 325, 335,
        336, 338, 342, 348, 359, 362, 365, 369, 371, 376, 382, 385, 386,
        392, 393, 398, 399, 401, 402, 404, 408, 416, 417, 420, 422, 423,
        428, 431, 433, 443, 448, 456, 457, 460, 470, 474, 483, 485, 486,
        488, 496, 504, 506, 512, 517, 520, 521, 528, 529, 538, 539, 543,
        544]),)

In [237]:
srcid = test_srcids_all[4]

In [239]:
pn_index = np.where(test_pn_srcids == srcid)[0]
mos_index = np.where(test_mos_srcids == srcid)[0]

In [240]:
pn_index

array([58])

In [241]:
mos_index

array([], dtype=int64)